# Experimentos - MLP do Zero

Este notebook registra os experimentos comparativos da implementação do MLP. Nesta primeira versão, uso um dataset sintético simples para comparar duas configurações antes de partir para o MNIST.

## Objetivo

Comparar duas arquiteturas diferentes usando o mesmo conjunto de dados, a mesma divisão de treino/validação e métricas de loss e acurácia. Isso ajuda a verificar se a rede está respondendo a mudanças de hiperparâmetros.

In [ ]:
from pathlib import Path
import sys

import numpy as np

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from mlp.network import MLP

## Dataset Sintético

O dataset abaixo tem duas classes separáveis. Ele é pequeno de propósito: se o MLP não aprender este problema, é melhor investigar a implementação antes de usar MNIST.

In [ ]:
X_train = np.array([
    [-1.0, -1.0],
    [-1.0, -0.5],
    [-0.5, -1.0],
    [-0.8, -0.7],
    [1.0, 1.0],
    [1.0, 0.5],
    [0.5, 1.0],
    [0.8, 0.7],
])
y_train = np.array([0, 0, 0, 0, 1, 1, 1, 1])

X_val = np.array([
    [-0.9, -0.8],
    [-0.6, -0.9],
    [0.9, 0.8],
    [0.6, 0.9],
])
y_val = np.array([0, 0, 1, 1])

X_train.shape, y_train.shape, X_val.shape, y_val.shape

## Configurações Comparadas

- Configuração A: rede menor, learning rate maior.
- Configuração B: rede maior, learning rate menor.

A comparação ainda não busca a melhor arquitetura final. O objetivo é confirmar que o fluxo de experimentos funciona.

In [ ]:
configurations = [
    {
        "nome": "Configuração A",
        "arquitetura": [2, 8, 4, 2],
        "learning_rate": 0.1,
        "batch_size": 4,
        "epochs": 40,
        "seed": 42,
    },
    {
        "nome": "Configuração B",
        "arquitetura": [2, 16, 8, 2],
        "learning_rate": 0.05,
        "batch_size": 4,
        "epochs": 40,
        "seed": 42,
    },
]

In [ ]:
def run_experiment(config):
    model = MLP(
        layer_sizes=config["arquitetura"],
        learning_rate=config["learning_rate"],
        seed=config["seed"],
    )
    history = model.fit(
        X_train,
        y_train,
        epochs=config["epochs"],
        batch_size=config["batch_size"],
        shuffle=True,
        X_val=X_val,
        y_val=y_val,
    )

    return {
        "nome": config["nome"],
        "arquitetura": " -> ".join(str(size) for size in config["arquitetura"]),
        "learning_rate": config["learning_rate"],
        "batch_size": config["batch_size"],
        "epochs": config["epochs"],
        "loss_final": history["loss"][-1],
        "accuracy_final": history["accuracy"][-1],
        "val_loss_final": history["val_loss"][-1],
        "val_accuracy_final": history["val_accuracy"][-1],
    }


results = [run_experiment(config) for config in configurations]
results

In [ ]:
header = (
    "Configuração | Arquitetura | LR | Batch | Épocas | "
    "Loss Treino | Acc Treino | Loss Val | Acc Val"
)
separator = "--- | --- | --- | --- | --- | --- | --- | --- | ---"
rows = []

for result in results:
    rows.append(
        "{nome} | {arquitetura} | {learning_rate} | {batch_size} | {epochs} | "
        "{loss_final:.6f} | {accuracy_final:.6f} | "
        "{val_loss_final:.6f} | {val_accuracy_final:.6f}".format(**result)
    )

print(header)
print(separator)
for row in rows:
    print(row)

## Observação

Como o dataset sintético é muito simples, as duas configurações devem alcançar desempenho alto. A comparação mais importante virá no MNIST, onde arquitetura, learning rate e batch size terão impacto mais visível.